# 线性回归的简洁实现


## 生成数据集

我们首先生成数据集。

In [1]:
import numpy as np
import torch
# 用于读取数据集, data.DataLoader
from torch.utils import data

In [2]:
"""生成y=Xw+b+噪声"""
def synthetic_data(true_w, true_b, num_examples):
    # 正态生成特征 X
    X = torch.normal(0, 1, (num_examples, len(true_w)))
    # 生成 真实的 Y
    y = torch.matmul(X, true_w) + true_b
    # 加入噪声
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape(-1, 1)

# 真实值
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

## 读取数据集

In [3]:
# 布尔值`is_train`表示是否希望数据迭代器对象在每个迭代周期内打乱数据。
def load_array(data_arrays, batch_size, is_train=True):
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

使用`data_iter`的方式与我们在 :numref:`sec_linear_scratch`中使用`data_iter`函数的方式相同。为了验证是否正常工作，让我们读取并打印第一个小批量样本。

这里我们使用`iter`构造Python迭代器，并使用`next`从迭代器中获取第一项。

In [4]:
next(iter(data_iter))

[tensor([[ 0.0600,  0.6358],
         [-1.2318, -0.5094],
         [-1.1597,  2.1589],
         [-1.8325, -0.3352],
         [ 1.0379, -0.5066],
         [-1.0586, -0.8972],
         [-1.6609, -0.4830],
         [ 1.3146, -0.4803],
         [ 0.8408, -1.5116],
         [ 0.5526, -0.6602]]),
 tensor([[ 2.1847],
         [ 3.4574],
         [-5.4713],
         [ 1.6689],
         [ 8.0047],
         [ 5.1358],
         [ 2.5027],
         [ 8.4684],
         [11.0212],
         [ 7.5447]])]

## 定义模型

在下面的例子中，我们的模型只包含一个层，因此实际上不需要`Sequential`。

但是由于以后几乎所有的模型都是多层的，在这里使用Sequential会让你熟悉“标准的流水线”。

In [5]:
# nn是神经网络的缩写
from torch import nn

# 第一个指定输入层神经元数
# 第二个指定输出层神经元数
net = nn.Sequential(nn.Linear(2, 1))

## 初始化模型参数

在使用`net`之前，我们需要初始化模型参数。如在线性回归模型中的权重和偏置。

深度学习框架通常有预定义的方法来初始化参数。在这里，我们指定
- 每个权重参数应该从均值为0、标准差为0.01的正态分布中随机采样，
- 偏置参数将初始化为零。


我们通过`net[0]`选择网络中的第一个图层，
然后使用`weight.data`和`bias.data`方法访问参数。
我们还可以使用替换方法`normal_`和`fill_`来重写参数值。

In [6]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

## 定义损失函数


[**计算均方误差使用的是`MSELoss`类，也称为平方$L_2$范数**]。
默认情况下，它返回所有样本损失的平均值。


In [8]:
loss = nn.MSELoss()

## 定义优化算法


小批量随机梯度下降算法是一种优化神经网络的标准工具，
PyTorch在`optim`模块中实现了该算法的许多变种。
当我们(**实例化一个`SGD`实例**)时，我们要指定优化的参数
（可通过`net.parameters()`从我们的模型中获得）以及优化算法所需的超参数字典。
小批量随机梯度下降只需要设置`lr`值，这里设置为0.03。


In [9]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

## 训练

通过深度学习框架的高级API来实现我们的模型只需要相对较少的代码。
我们不必单独分配参数、不必定义我们的损失函数，也不必手动实现小批量随机梯度下降。
当我们需要更复杂的模型时，高级API的优势将大大增加。
当我们有了所有的基本组件，[**训练过程代码与我们从零开始实现时所做的非常相似**]。

回顾一下：在每个迭代周期里，我们将完整遍历一次数据集（`train_data`），
不停地从中获取一个小批量的输入和相应的标签。
对于每一个小批量，我们会进行以下步骤:

* 通过调用`net(X)`生成预测并计算损失`l`（前向传播）。
* 通过进行反向传播来计算梯度。
* 通过调用优化器来更新模型参数。

为了更好的衡量训练效果，我们计算每个迭代周期后的损失，并打印它来监控训练过程。


In [10]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000173
epoch 2, loss 0.000101
epoch 3, loss 0.000101


下面我们[**比较生成数据集的真实参数和通过有限数据训练获得的模型参数**]。
要访问参数，我们首先从`net`访问所需的层，然后读取该层的权重和偏置。
正如在从零开始实现中一样，我们估计得到的参数与生成数据的真实参数非常接近。


In [11]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： tensor([0.0008, 0.0002])
b的估计误差： tensor([0.0002])
